# DARs (marker regions and disease)

In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(repr)
    library(patchwork)
    library(ggplot2)
    library(Signac)
    library(tidyverse)
    library(GenomicRanges)
    library(edgeR)
    library(SingleCellExperiment)
    library(Matrix)
    library(scran)
    library(scater)
    library(ggrepel)
})
options(future.globals.maxSize = Inf)
options(Seurat.object.assay.version = "v5")
options(ggrepel.max.overlaps = Inf)

In [2]:
setwd("/tscc/projects/ps-epigen/users/biy022/MGH/data/tissue/")

In [3]:
MGH_All <- readRDS("combined.all.atac/MGH.ALL.rds")

In [4]:
downsampled_barcodes <- read.table(
    "region_combined_analysis/scenicplus/all//MGH_barcodes.tsv",
    header = FALSE, sep = "\t")

In [5]:
MGH_dsampl <- MGH_All[, downsampled_barcodes[, 1]]

In [6]:
MGH_dsampl

An object of class Seurat 
654680 features across 124380 samples within 2 assays 
Active assay: RNA (36601 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: ATAC
 4 dimensional reductions calculated: pca, umap.default, harmony, umap.harmony

## Subclass marker regions

In [7]:
MGH_counts <- LayerData(MGH_dsampl, assay = "ATAC", layer = c("counts"))
MGH_meta <- MGH_dsampl@meta.data

In [8]:
out_dir <- "region_combined_analysis/dar_results/all/marker_regions/"

In [9]:
edgeR_test_subclass <- function(counts_mtx, meta_df, label) {
    
    meta_df$region_label <- "BG"
    meta_df$region_label[meta_df$Subclass == label] <- "FG"
    meta_df$group_ident <- paste(meta_df$region_label, meta_df$orig.ident, sep = "-")
    curr_counts <- t(rowsum(t(counts_mtx), group = meta_df$group_ident))
    comp_group <- sapply(str_split(colnames(curr_counts), pattern = "-"), `[`, 1)
    comp_group <- factor(comp_group, levels = c("BG", "FG"))
    y <- DGEList(
        counts = curr_counts, 
        group = comp_group,
        remove.zeros = TRUE
    )
    
    # print(label)
    # print(nrow(y$counts))
    # keep <- filterByExpr(y, min.count = 5, min.prop = 0.5, large.n = 5)
    # y <- y[keep, , keep.lib.sizes = FALSE]
    # print(nrow(y$counts))
    # flush.console()
    # if (nrow(y$counts) < 5) {
    #     return(results)
    # }
    y <- calcNormFactors(y)
    design <- model.matrix(~ 0 + group, data = y$samples)
    colnames(design) <- levels(y$samples$group)
    y <- estimateDisp(y, design = design, robust = TRUE)
    fit <- glmQLFit(y, design = design)
    qlf <- glmQLFTest(fit, contrast = c(-1, 1))
    
    test_results <- topTags(qlf, n = Inf, sort.by = "none")$table %>%
        dplyr::mutate(Region = rownames(.), Subclass = label)
    
    write.table(
        test_results,
        paste0(out_dir, gsub(x = label, pattern = "[ /]", replacement = "_"), ".tsv"),
        col.names = TRUE, quote = FALSE, sep = "\t", row.names = FALSE
    )
    
    hits <- test_results[test_results$FDR < 0.01 & test_results$logFC > 0, ]
    hits_bed <- as.data.frame(do.call(rbind, str_split(hits$Region, pattern = "-")))
    write.table(
        hits_bed,
        paste0(out_dir, gsub(x = label, pattern = "[ /]", replacement = "_"), ".bed"),
        col.names = FALSE, quote = FALSE, sep = "\t", row.names = FALSE
    )
}

In [10]:
for (label in unique(MGH_meta$Subclass)) {
    edgeR_test_subclass(MGH_counts, MGH_meta, label)
}

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts

Removing 10 rows with all zero counts



In [11]:
out_dir <- "region_combined_analysis/dar_results/all/disease_dars/"

In [12]:
edgeR_test_disease <- function(counts_mtx, meta_df, label) {
    
    curr_mtx <- counts_mtx[, meta_df$Subclass == label]
    curr_meta <- meta_df[meta_df$Subclass == label, ]
    curr_meta$group_ident <- paste(curr_meta$AD, curr_meta$orig.ident, sep = "-")
    curr_counts <- t(rowsum(t(curr_mtx), group = curr_meta$group_ident))
    comp_group <- sapply(str_split(colnames(curr_counts), pattern = "-"), `[`, 1)
    comp_group <- factor(comp_group, levels = c("Control", "AD"))
    y <- DGEList(
        counts = curr_counts, 
        group = comp_group,
        remove.zeros = TRUE
    )
    
    # print(label)
    # print(nrow(y$counts))
    # keep <- filterByExpr(y, min.count = 5, min.prop = 0.5, large.n = 5)
    # y <- y[keep, , keep.lib.sizes = FALSE]
    # print(nrow(y$counts))
    # flush.console()
    # if (nrow(y$counts) < 5) {
    #     return(results)
    # }
    y <- calcNormFactors(y)
    design <- model.matrix(~ 0 + group, data = y$samples)
    colnames(design) <- levels(y$samples$group)
    y <- estimateDisp(y, design = design, robust = TRUE)
    fit <- glmQLFit(y, design = design)
    qlf <- glmQLFTest(fit, contrast = c(-1, 1))
    
    test_results <- topTags(qlf, n = Inf, sort.by = "none")$table %>%
        dplyr::mutate(Region = rownames(.), Subclass = label)
    
    write.table(
        test_results,
        paste0(out_dir, gsub(x = label, pattern = "[ /]", replacement = "_"), ".tsv"),
        col.names = TRUE, quote = FALSE, sep = "\t", row.names = FALSE
    )
    
    uphits <- test_results[test_results$PValue < 0.01 & test_results$logFC > 0, ]
    uphits_bed <- as.data.frame(do.call(rbind, str_split(uphits$Region, pattern = "-")))
    write.table(
        uphits_bed,
        paste0(out_dir, gsub(x = label, pattern = "[ /]", replacement = "_"), "-AD.bed"),
        col.names = FALSE, quote = FALSE, sep = "\t", row.names = FALSE
    )
    
    downhits <- test_results[test_results$PValue < 0.01 & test_results$logFC < 0, ]
    downhits_bed <- as.data.frame(do.call(rbind, str_split(downhits$Region, pattern = "-")))
    write.table(
        downhits_bed,
        paste0(out_dir, gsub(x = label, pattern = "[ /]", replacement = "_"), "-CTRL.bed"),
        col.names = FALSE, quote = FALSE, sep = "\t", row.names = FALSE
    )
}

In [13]:
for (label in unique(MGH_meta$Subclass)) {
    edgeR_test_disease(MGH_counts, MGH_meta, label)
}

Removing 583179 rows with all zero counts

Removing 401686 rows with all zero counts

Removing 524076 rows with all zero counts

Removing 439164 rows with all zero counts

Removing 415063 rows with all zero counts

Removing 582666 rows with all zero counts

Removing 446888 rows with all zero counts

Removing 462437 rows with all zero counts

Removing 480869 rows with all zero counts

Removing 473553 rows with all zero counts

Removing 485439 rows with all zero counts

Removing 513062 rows with all zero counts

Removing 413152 rows with all zero counts

Removing 546033 rows with all zero counts

Removing 477109 rows with all zero counts

Removing 528102 rows with all zero counts

Removing 483447 rows with all zero counts

Removing 432218 rows with all zero counts

Removing 522054 rows with all zero counts

Removing 502909 rows with all zero counts

Removing 488446 rows with all zero counts

Removing 607891 rows with all zero counts

Removing 533992 rows with all zero counts

Removing 61